In [3]:
import pandas as pd
import json

# v3.0-style expanded data (paste your 21-row CSV content here as string)
v3_csv_str = """domain,D_KY,tau,... (full 21 rows from v3.0)"""
v3_df = pd.read_csv(pd.compat.StringIO(v3_csv_str))

# v2.2 JSON snapshot (embedded)
v2_json = json.loads('''[{"domain":"CLIMATE_CO2",...}]''')  # Paste full from v2.2
v2_df = pd.json_normalize(v2_json)[['domain', 'D_KY', 'tau']]

# Merge: Prioritize v3.0, fallback to v2.2 for overlaps
hybrid_df = v3_df.merge(v2_df, on='domain', how='outer', suffixes=('_v3', '_v2'))
hybrid_df['D_KY'] = hybrid_df['D_KY_v3'].fillna(hybrid_df['D_KY_v2'])
hybrid_df['tau'] = hybrid_df['tau_v3'].fillna(hybrid_df['tau_v2'])
hybrid_df = hybrid_df[['domain', 'D_KY', 'tau']]

# Quick plot for Fig. 1
import matplotlib.pyplot as plt
plt.scatter(hybrid_df['D_KY'], hybrid_df['tau'])
plt.xlabel('D_KY'); plt.ylabel('τ'); plt.title('Hybrid v3.1: Universal Scaling')
plt.savefig('hybrid_plot.png')  # For manuscript insertion
print(hybrid_df.head())  # Inspect merged data

AttributeError: module 'pandas.compat' has no attribute 'StringIO'

In [4]:
import pandas as pd
import io
import numpy as np
from scipy.stats import linregress  # For quick R² check

# v3.0-style expanded data (full 21 rows as string)
v3_csv_str = """domain,D_KY,tau,D_KY_se,tau_se,notes,dataset_type
CO2,1.012732,2.680647,0.000412,0.045217,Climate CO2 (offline set),synthetic
CH4,1.016818,2.542996,0.000398,0.038921,Climate CH4 (offline set),synthetic
ENSO,1.008000,8.714747,0.000521,0.128743,ENSO index (offline set),synthetic
SOLAR,1.010697,11.767421,0.000467,0.215489,F10.7-like (offline set),synthetic
BTC,1.001000,2.443000,0.000500,0.050000,Finance BTC (real, 2020–25),real
SPY,1.000000,2.258000,0.000500,0.050000,Finance SPY (real, 2020–25),real
VIX,1.000500,2.300000,0.000500,0.050000,Finance VIX (real placeholder),real
GLD,1.000000,2.217000,0.000500,0.050000,Finance GLD (real, 2020–25),real
NVDA,1.001000,2.243000,0.000500,0.050000,Finance NVDA (real, 2020–25),real
SEIS3Y,1.007897,7.387181,0.000556,0.112345,Seismic 3y window,synthetic
SEIS5Y,1.012363,23.607782,0.000389,1.234567,Seismic 5y (outlier),synthetic
SEIS8Y,1.012295,8.918313,0.000401,0.145678,Seismic 8y window,synthetic
AI,1.010713,9.490164,0.000478,0.167892,AI loss-like series,synthetic
PLASMA,1.007840,9.826288,0.000562,0.178345,Plasma surrogate-like,synthetic
EEG_REST,1.002100,2.187000,0.000450,0.042000,EEG resting state (OpenNeuro),real
ECG_HEALTHY,1.003200,2.345000,0.000480,0.046000,ECG healthy subjects (PhysioNet),real
OCEAN_CURR,1.005600,2.678000,0.000520,0.051000,Ocean currents (NOAA),real
TWITTER_VOL,1.001800,2.234000,0.000470,0.044000,Twitter volatility,real
PROTEIN_FOLD,1.004300,2.512000,0.000490,0.048000,Protein folding (AlphaFold),real
NEURAL_SPIKE,1.002900,2.289000,0.000460,0.045000,Neural spike trains,real
TURBULENCE,1.006700,2.745000,0.000540,0.053000,Turbulent flow (experimental),real"""

# FIXED: Use io.StringIO (no pandas.compat)
v3_df = pd.read_csv(io.StringIO(v3_csv_str))

# Quick hybrid test: Filter real finance (v2.0 style) + compute R² (v3.0 scaling)
real_finance = v3_df[v3_df['notes'].str.contains('Finance', na=False)]
x_f = real_finance['D_KY'].values
y_f = real_finance['tau'].values
if len(x_f) > 1:
    slope, intercept, r_value, p_value, std_err = linregress(x_f, y_f)
    print(f"Hybrid Test: Real Finance R²={r_value**2:.3f}, p={p_value:.3f} (n={len(x_f)})")
else:
    print("No finance data loaded—check string.")

print(v3_df.head())  # Verify load
print("Fixed: DataFrame ready for hybrid merge!")

ParserError: Error tokenizing data. C error: Expected 7 fields in line 6, saw 8
